In [1]:

import numpy as np
import plotly.graph_objects as go
import os
import webbrowser

np.random.seed(42)
fig = go.Figure()

# ==========================================
# FUNKCJE GEOMETRYCZNE
# ==========================================

def sphere(center, scale, color, name, opacity=1):
    u = np.linspace(0, 2*np.pi, 35)
    v = np.linspace(0, np.pi, 25)

    x = center[0] + scale[0]*np.outer(np.cos(u), np.sin(v))
    y = center[1] + scale[1]*np.outer(np.sin(u), np.sin(v))
    z = center[2] + scale[2]*np.outer(np.ones_like(u), np.cos(v))

    fig.add_trace(go.Surface(
        x=x, y=y, z=z,
        surfacecolor=np.zeros_like(x),
        colorscale=[[0,color],[1,color]],
        showscale=False,
        opacity=opacity,
        name=name,
        hovertemplate=name+"<extra></extra>",
        lighting=dict(
            ambient=0.45,
            diffuse=0.9,
            specular=0.9,
            roughness=0.3
        )
    ))

def cone(start, end, radius, color, name):
    a = np.array(start, dtype=float)
    b = np.array(end, dtype=float)

    d = b-a
    d = d/np.linalg.norm(d)

    ref = np.array([0.,1.,0.])
    if abs(np.dot(d,ref)) > 0.95:
        ref = np.array([1.,0.,0.])

    u = np.cross(d,ref)
    u /= np.linalg.norm(u)
    v = np.cross(d,u)

    angles = np.linspace(0,2*np.pi,24)
    vertices = [a + radius*(np.cos(t)*u+np.sin(t)*v)
                for t in angles]
    vertices.append(b)

    n = len(angles)
    ii = list(range(n))
    jj = [(i+1)%n for i in range(n)]
    kk = [n]*n

    fig.add_trace(go.Mesh3d(
        x=[p[0] for p in vertices],
        y=[p[1] for p in vertices],
        z=[p[2] for p in vertices],
        i=ii, j=jj, k=kk,
        color=color,
        name=name,
        hovertemplate=name+"<extra></extra>",
        flatshading=True
    ))

def torus(center, major, minor, color, name,
          vertical=False, opacity=1):
    u = np.linspace(0,2*np.pi,90)
    v = np.linspace(0,2*np.pi,24)
    U,V = np.meshgrid(u,v)

    if vertical:
        X = center[0]+(major+minor*np.cos(V))*np.cos(U)
        Y = center[1]+minor*np.sin(V)
        Z = center[2]+(major+minor*np.cos(V))*np.sin(U)
    else:
        X = center[0]+(major+minor*np.cos(V))*np.cos(U)
        Y = center[1]+(major+minor*np.cos(V))*np.sin(U)
        Z = center[2]+minor*np.sin(V)

    fig.add_trace(go.Surface(
        x=X,y=Y,z=Z,
        colorscale=[[0,color],[1,color]],
        showscale=False,
        opacity=opacity,
        name=name,
        hoverinfo="skip"
    ))

# ==========================================
# 1. PLANETA – PROCEDURALNY KRAJOBRAZ
# ==========================================

x = np.linspace(-7,7,130)
y = np.linspace(-7,7,130)
X,Y = np.meshgrid(x,y)

R = np.sqrt(X**2+Y**2)

Z = (
    0.25*np.sin(X*1.5)*np.cos(Y*1.3)
    + 0.35*np.sin(X*0.7+Y*0.8)
    + 2.1*np.exp(-((X-2)**2+(Y-1)**2)/2.2)
    + 1.5*np.exp(-((X+2.8)**2+(Y+1.5)**2)/1.5)
    + 1.1*np.exp(-((X+0.5)**2+(Y-3)**2)/1.2)
)

Z -= 0.025*R**2

fig.add_trace(go.Surface(
    x=X,y=Y,z=Z,
    surfacecolor=Z,
    colorscale=[
        [0,"#10102C"],
        [0.25,"#27245F"],
        [0.5,"#46318C"],
        [0.75,"#168B9B"],
        [1,"#65F5D4"]
    ],
    showscale=False,
    name="Powierzchnia planety",
    hovertemplate="Tajemniczy teren<extra></extra>"
))

# ==========================================
# 2. LATAJĄCE WYSPY
# ==========================================

islands = [
    (-4,1,3.3,1.3),
    (4,2,4.0,1.0),
    (2,-4,3.5,1.1),
    (-3,-4,2.9,0.8)
]

for px,py,pz,s in islands:
    sphere(
        (px,py,pz),
        (s,s*0.8,s*0.35),
        "#51408C",
        "Latająca wyspa"
    )

    # Wieża na wyspie
    cone(
        (px,py,pz+0.1),
        (px,py,pz+1.3*s),
        0.22*s,
        "#39DCEB",
        "Wieża"
    )

# ==========================================
# 3. WIELKI PORTAL
# ==========================================

torus(
    (0,-2.5,2.5),
    1.35,
    0.16,
    "#00FFFF",
    "Portal międzywymiarowy",
    vertical=True
)

torus(
    (0,-2.5,2.5),
    1.12,
    0.055,
    "#D34BFF",
    "Magiczny pierścień",
    vertical=True
)

sphere(
    (0,-2.5,2.5),
    (1.1,0.04,1.1),
    "#38206E",
    "Wnętrze portalu",
    opacity=0.35
)

# ==========================================
# 4. GIGANTYCZNE KRYSZTAŁY
# ==========================================

crystals = [
    (-3,0,1.2,1.7),
    (3,1,1.1,2.1),
    (-1,4,0.7,1.3),
    (4,-2,0.6,1.4),
    (-4,-2,0.6,1.1),
    (1,3,0.7,1.5)
]

for px,py,pz,h in crystals:
    cone(
        (px,py,pz),
        (px+0.15,py,pz+h),
        0.35,
        "#36E8F5",
        "Kryształ energii"
    )

    sphere(
        (px+0.15,py,pz+h),
        (0.12,0.12,0.12),
        "#B5FFFF",
        "Świetlisty kryształ"
    )

# ==========================================
# 5. KOSMICZNY STATEK
# ==========================================

# Kadłub
sphere(
    (2.5,0,5.1),
    (1.0,0.35,0.32),
    "#C8D9E9",
    "Statek kosmiczny"
)

# Nos statku
cone(
    (3.2,0,5.1),
    (4.2,0,5.1),
    0.28,
    "#F5F5FF",
    "Nos statku"
)

# Skrzydła
sphere(
    (2.5,0.65,5.0),
    (0.65,0.12,0.12),
    "#6376C8",
    "Skrzydło"
)

sphere(
    (2.5,-0.65,5.0),
    (0.65,0.12,0.12),
    "#6376C8",
    "Skrzydło"
)

# Silniki
for yy in [-0.18,0.18]:
    sphere(
        (1.65,yy,5.1),
        (0.18,0.13,0.13),
        "#FF6A00",
        "Silnik"
    )

# ==========================================
# 6. PIERŚCIENIE PLANETY
# ==========================================

torus(
    (0,0,-3),
    6.5,
    0.10,
    "#D19AFF",
    "Pierścień planety",
    opacity=0.8
)

torus(
    (0,0,-3.15),
    7.0,
    0.07,
    "#45C7E8",
    "Drugi pierścień",
    opacity=0.7
)

# ==========================================
# 7. KSIĘŻYC
# ==========================================

sphere(
    (5,4,5.5),
    (0.65,0.65,0.65),
    "#B5A7D8",
    "Księżyc"
)

# ==========================================
# 8. KOSMICZNE GWIAZDY
# ==========================================

n = 900

sx = np.random.uniform(-10,10,n)
sy = np.random.uniform(-10,10,n)
sz = np.random.uniform(2,12,n)

fig.add_trace(go.Scatter3d(
    x=sx,y=sy,z=sz,
    mode="markers",
    marker=dict(
        size=np.random.uniform(1,3,n),
        color=np.random.choice(
            ["#FFFFFF","#A6E8FF","#D9A5FF"],
            n
        ),
        opacity=0.85
    ),
    name="Gwiazdy",
    hoverinfo="skip"
))

# ==========================================
# 9. NAPISY W SCENIE
# ==========================================

fig.add_trace(go.Scatter3d(
    x=[0,0,2.5],
    y=[0,-2.5,0],
    z=[6.5,4.5,6.0],
    mode="text",
    text=[
        "NEBULA",
        "PORTAL",
        "SPACECRAFT"
    ],
    textfont=dict(
        color=["#FFFFFF","#00FFFF","#FFD700"],
        size=[22,14,12]
    ),
    showlegend=False,
    hoverinfo="skip"
))

# ==========================================
# 10. WYGLĄD SCENY
# ==========================================

fig.update_layout(
    title=dict(
        text="✦ NEBULA – THE LOST WORLD ✦",
        x=0.5,
        font=dict(size=27,color="#FFFFFF")
    ),

    paper_bgcolor="#050510",
    font=dict(color="white"),

    scene=dict(
        bgcolor="#050510",

        xaxis=dict(
            visible=False,
            range=[-9,9]
        ),
        yaxis=dict(
            visible=False,
            range=[-9,9]
        ),
        zaxis=dict(
            visible=False,
            range=[-4,12]
        ),

        aspectmode="manual",
        aspectratio=dict(
            x=1.2,
            y=1.2,
            z=0.9
        ),

        camera=dict(
            eye=dict(x=1.7,y=1.7,z=1.1)
        ),

        dragmode="orbit"
    ),

    height=900,
    margin=dict(l=0,r=0,t=75,b=0),
    showlegend=True
)

# ==========================================
# 11. ZAPIS DO HTML
# ==========================================

filename = os.path.abspath("NEBULA_WORLD.html")

fig.write_html(
    filename,
    include_plotlyjs=True,
    full_html=True,
    auto_open=False
)

webbrowser.open("file://" + filename)

print("NEBULA – GOTOWE!")
print("Plik:",filename)
print("Obracaj scenę myszką i używaj scrolla.")


NEBULA – GOTOWE!
Plik: c:\Users\lab812-03.LABORATORIUM\Documents\GitHub\testMK\NEBULA_WORLD.html
Obracaj scenę myszką i używaj scrolla.
